#--------------------------------------
#CATALOGO

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS electrocasa;

#--------------------------------------
#ESQUEMAS

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electrocasa.bronze;

CREATE SCHEMA IF NOT EXISTS electrocasa.silver;

CREATE SCHEMA IF NOT EXISTS electrocasa.gold;

#--------------------------------------
#VOLUMEN

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS electrocasa.bronze.landing;

#--------------------------------------
#CARPETAS DE LANDING

In [0]:
from pathlib import Path

BASE_LANDING = Path("/Volumes/electrocasa/bronze/landing")

carpetas = [
    "ventas",
    "catalogo",
    "empleados",
    "resenas",
    "devoluciones"
]

for carpeta in carpetas:
    ruta = BASE_LANDING / carpeta
    ruta.mkdir(parents=True, exist_ok=True)
    print(f"OK: {ruta}")

#--------------------------------------
#PERMISOS

In [0]:
%sql
GRANT USE CATALOG
ON CATALOG electrocasa
TO `Ingenieria`;

GRANT USE SCHEMA
ON SCHEMA electrocasa.bronze
TO `Ingenieria`;

GRANT USE SCHEMA
ON SCHEMA electrocasa.silver
TO `Ingenieria`;

GRANT USE SCHEMA
ON SCHEMA electrocasa.gold
TO `Ingenieria`;

GRANT SELECT, MODIFY
ON SCHEMA electrocasa.bronze
TO `Ingenieria`;

GRANT SELECT, MODIFY
ON SCHEMA electrocasa.silver
TO `Ingenieria`;

GRANT SELECT, MODIFY
ON SCHEMA electrocasa.gold
TO `Ingenieria`;

In [0]:
%sql
GRANT USE CATALOG
ON CATALOG electrocasa
TO `Analistas`;

GRANT USE SCHEMA
ON SCHEMA electrocasa.gold
TO `Analistas`;

GRANT SELECT
ON SCHEMA electrocasa.gold
TO `Analistas`;

In [0]:
%sql
GRANT USE CATALOG
ON CATALOG electrocasa
TO `Auditoria`;

GRANT BROWSE
ON CATALOG electrocasa
TO `Auditoria`;

GRANT USE SCHEMA
ON SCHEMA electrocasa.gold
TO `Auditoria`;

GRANT SELECT
ON SCHEMA electrocasa.gold
TO `Auditoria`;

#--------------------------------------
#MASKING

In [0]:
%sql
DESCRIBE FUNCTION mask;

In [0]:
%sql
CREATE OR REPLACE FUNCTION electrocasa.silver.mask_dni(valor STRING)
RETURNS STRING
RETURN CASE
    WHEN is_account_group_member('Ingenieria')
        THEN valor
    ELSE mask(valor)
END;

In [0]:
%sql
CREATE OR REPLACE VIEW electrocasa.gold.empleados_activos_seguro AS
SELECT
    id_empleado,
    nombre,

    CASE
        WHEN is_account_group_member('Ingenieria')
            THEN dni
        ELSE mask(dni)
    END AS dni,

    email,

    CASE
        WHEN is_account_group_member('Ingenieria')
            THEN CAST(salario AS STRING)
        ELSE mask(CAST(salario AS STRING))
    END AS salario,

    sucursal_id,
    cargo,
    tipo_evento,
    fecha_evento,
    fecha_inicio,
    fecha_fin,
    es_actual

FROM electrocasa.silver.empleados_silver;

In [0]:
%sql
GRANT SELECT
ON VIEW electrocasa.gold.empleados_activos_seguro
TO `Analistas`;

GRANT SELECT
ON VIEW electrocasa.gold.empleados_activos_seguro
TO `Auditoria`;

#--------------------------------------
#VERIFICAR CATALOGOS, SCHEMAS, VOLUMENES, PERMISOS, ETC

In [0]:
%sql
SHOW SCHEMAS IN electrocasa;

In [0]:
%sql
SHOW VOLUMES IN electrocasa.bronze;

In [0]:
%sql
SHOW GRANTS ON CATALOG electrocasa;

In [0]:
%sql
SHOW GRANTS ON SCHEMA electrocasa.bronze;
SHOW GRANTS ON SCHEMA electrocasa.silver;
SHOW GRANTS ON SCHEMA electrocasa.gold;

In [0]:
from pathlib import Path

BASE_LANDING = Path("/Volumes/electrocasa/bronze/landing")

for carpeta in sorted(BASE_LANDING.iterdir()):
    if carpeta.is_dir():
        print(f"OK: {carpeta.name}/")